# Feature Engineering Models

## Objective
In this notebook, I build a feature-based forecasting pipeline using the cleaned time series from Week 1.

The goal is to transform the original time series into a structured dataset with engineered features such as calendar variables, lag features, rolling statistics, and external signals like oil prices and holidays.

The models in this notebook will be trained on data from **2013-01-01 to 2013-12-31** and evaluated on data from **2014-01-01 to 2014-03-31**.

# 1. Environment Setup

In [ ]:
# Import the libraries
import pandas as pd
import matplotlib.pyplot as plt

# 2. Data Loading

In [ ]:
# Load the cleaned main time series from Week 1
timeseries = pd.read_csv("../../data/timeseries_cleaned.csv")

# Load the processed oil dataset prepared in the previous notebook
oil = pd.read_csv("../../data/processed/oil_interpolated.csv")

# Load the holidays dataset from the raw data folder
holidays = pd.read_csv("../../data/raw/holidays.csv")

# 3. Data Exploration

In [ ]:
# Display the first rows of each dataset
display(timeseries.head())
display(oil.head())
display(holidays.head())

In [ ]:
# Check the shape of each dataset
print("Timeseries shape:", timeseries.shape)
print("Oil shape:", oil.shape)
print("Holidays shape:", holidays.shape)

In [ ]:
# Check basic information about each dataset
timeseries.info()
oil.info()
holidays.info()

In [ ]:
# Convert date columns to datetime format
timeseries["date"] = pd.to_datetime(timeseries["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

In [ ]:
# Check data types after datetime conversion
print(timeseries.dtypes)
print(oil.dtypes)
print(holidays.dtypes)

In [ ]:
# Check the date range of each dataset
print("Timeseries:", timeseries["date"].min(), "to", timeseries["date"].max())
print("Oil:", oil["date"].min(), "to", oil["date"].max())
print("Holidays:", holidays["date"].min(), "to", holidays["date"].max())

In [ ]:
# Check missing values in each dataset
print("Timeseries missing values:")
print(timeseries.isna().sum())

print("\nOil missing values:")
print(oil.isna().sum())

print("\nHolidays missing values:")
print(holidays.isna().sum())

In [ ]:
# Plot unit sales over time
plt.figure(figsize=(14, 5))
plt.plot(df["date"], df["unit_sales"])
plt.title("Unit Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Unit Sales")
plt.show()

# 4. Feature Engineering

In [ ]:
# Create a working copy of the main time series
df = timeseries.copy()

In [ ]:
# Display the first rows of the working dataframe
df.head()

In [ ]:
# Check the shape and columns of the working dataframe
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## Calendar Features

These features are created directly from the date column. They help capture recurring calendar patterns such as weekdays, weekends, month boundaries, and seasonal position within the year.

In [ ]:
# Create basic calendar features from the date column
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

df.head()

In [ ]:
# Create additional calendar features
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df.head()

In [ ]:
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

df.head()

## Lag Features

Lag features use past sales values as predictors for the current day. They help the model capture short-term memory and repeating temporal patterns in the time series.

In [ ]:
# Create lag features based on past sales values
df["lag_1"] = df["unit_sales"].shift(1)
df["lag_7"] = df["unit_sales"].shift(7)
df["lag_14"] = df["unit_sales"].shift(14)
df["lag_30"] = df["unit_sales"].shift(30)

In [ ]:
# Display the new lag features
df[["date", "unit_sales", "lag_1", "lag_7", "lag_14", "lag_30"]].head(35)

## Rolling Features

Rolling features summarize recent sales behavior over fixed windows. They help the model capture short-term trends and local variability in the time series.

In [ ]:
# Create rolling statistics based on past sales values
df["rolling_7d_mean"] = df["unit_sales"].shift(1).rolling(window=7).mean()
df["rolling_14d_mean"] = df["unit_sales"].shift(1).rolling(window=14).mean()
df["rolling_30d_mean"] = df["unit_sales"].shift(1).rolling(window=30).mean()
df["rolling_7d_std"] = df["unit_sales"].shift(1).rolling(window=7).std()

In [ ]:
# Display the new rolling features
df[[
    "date",
    "unit_sales",
    "rolling_7d_mean",
    "rolling_14d_mean",
    "rolling_30d_mean",
    "rolling_7d_std"
]].head(35)

## Oil Features

Oil-related features are added as external signals. They may help capture broader economic conditions that influence sales behavior over time.

In [ ]:
# Merge the working dataframe with the processed oil dataset
df = df.merge(
    oil[["date", "oil_price_interpolated"]],
    on="date",
    how="left"
)

In [ ]:
df[["date", "unit_sales", "oil_price_interpolated"]].head()

In [ ]:
# Fill missing oil values after the merge
df["oil_price_interpolated"] = df["oil_price_interpolated"].ffill()

In [ ]:
# Check missing values in the merged oil feature
df["oil_price_interpolated"].isna().sum()

In [ ]:
# Create oil-based features
df["oil_lag_1"] = df["oil_price_interpolated"].shift(1)
df["oil_rolling_7d_mean"] = df["oil_price_interpolated"].shift(1).rolling(window=7).mean()

In [ ]:
# Display the oil-based features
df[["date", "oil_price_interpolated", "oil_lag_1", "oil_rolling_7d_mean"]].head(10)

## Holiday Features

Holiday features are added as binary indicators. They help the model detect whether special dates such as national, regional, or local holidays may influence sales.

In [ ]:
# Create separate holiday flags by locale type
national_holidays = holidays[holidays["locale"] == "National"][["date"]].copy()
regional_holidays = holidays[holidays["locale"] == "Regional"][["date"]].copy()
local_holidays = holidays[holidays["locale"] == "Local"][["date"]].copy()

In [ ]:
national_holidays["is_national_holiday"] = 1
regional_holidays["is_regional_holiday"] = 1
local_holidays["is_local_holiday"] = 1

In [ ]:
# Merge the national holiday flag into the working dataframe
df = df.merge(
    national_holidays,
    on="date",
    how="left"
)

In [ ]:
# Merge the regional holiday flag into the working dataframe
df = df.merge(
    regional_holidays,
    on="date",
    how="left"
)

In [ ]:
# Merge the local holiday flag into the working dataframe
df = df.merge(
    local_holidays,
    on="date",
    how="left"
)

In [ ]:
# Fill missing holiday flags with 0
df["is_national_holiday"] = df["is_national_holiday"].fillna(0)
df["is_regional_holiday"] = df["is_regional_holiday"].fillna(0)
df["is_local_holiday"] = df["is_local_holiday"].fillna(0)

In [ ]:
# Convert holiday flags to integer type
df["is_national_holiday"] = df["is_national_holiday"].astype(int)
df["is_regional_holiday"] = df["is_regional_holiday"].astype(int)
df["is_local_holiday"] = df["is_local_holiday"].astype(int)

In [ ]:
# Display the holiday flags in the working dataframe
df[["date", "is_national_holiday", "is_regional_holiday", "is_local_holiday"]].head(10)

In [ ]:
# Check the final list of columns after feature engineering
print(df.columns.tolist())

In [ ]:
# Check missing values after creating all features
df.isna().sum()

In [ ]:
# Remove rows with missing values created by feature engineering
df = df.dropna().copy()

In [ ]:
# Check the final shape of the feature-engineered dataframe
print("Final dataframe shape:", df.shape)

In [ ]:
# Check that no missing values remain after dropping incomplete rows
print("Remaining missing values:", df.isna().sum().sum())

In [ ]:
# Display the final feature-engineered dataframe
df.head()

In [ ]:
# Save the final feature-engineered dataframe
df.to_csv("../../data/processed/feature_engineered_timeseries.csv", index=False)

## Feature Engineering Summary

At this stage, the time series has been transformed into a structured feature-based dataset.
The final dataframe is clean, contains no missing values, and is ready for the next step: train-test split and model training.